## Image prediction analysis

In [4]:
import torch

data = torch.load("image_with_side_task.pth", map_location='cpu')
print(type(data))
print(data.keys() if isinstance(data, dict) else data)

<class 'dict'>
dict_keys(['config', 'epoch', 'cur_step', 'best_valid_score', 'state_dict', 'other_parameter', 'optimizer'])


In [5]:
from recbole.utils.utils import init_seed
from recbole.utils.logger import init_logger
from logging import getLogger

config = data['config']
config.final_config_dict['data_path'] = '/Users/jakubmalczak/UNI/INŻ/SequentialRecommendation/recbole/data/dataset/Amazon_Sports_and_Outdoors'
config['device'] = 'cpu'

for attr, value in config.__dict__.items():
    print(f"{attr} = {value}")

init_seed(config['seed'], config['reproducibility'])
init_logger(config)
logger = getLogger()
logger.info(config)

parameters = {'General': ['gpu_id', 'use_gpu', 'seed', 'reproducibility', 'state', 'data_path', 'benchmark_filename', 'show_progress', 'config_file', 'save_dataset', 'save_dataloaders'], 'Training': ['epochs', 'train_batch_size', 'learner', 'learning_rate', 'training_neg_sample_num', 'training_neg_sample_distribution', 'eval_step', 'stopping_step', 'checkpoint_dir', 'clip_grad_norm', 'loss_decimal_place', 'weight_decay'], 'Evaluation': ['eval_args', 'metrics', 'topk', 'valid_metric', 'valid_metric_bigger', 'eval_batch_size', 'metric_decimal_place'], 'Dataset': ['field_separator', 'seq_separator', 'USER_ID_FIELD', 'ITEM_ID_FIELD', 'RATING_FIELD', 'TIME_FIELD', 'seq_len', 'LABEL_FIELD', 'threshold', 'NEG_PREFIX', 'ITEM_LIST_LENGTH_FIELD', 'LIST_SUFFIX', 'MAX_ITEM_LIST_LENGTH', 'POSITION_FIELD', 'HEAD_ENTITY_ID_FIELD', 'TAIL_ENTITY_ID_FIELD', 'RELATION_ID_FIELD', 'ENTITY_ID_FIELD', 'load_col', 'unload_col', 'unused_col', 'additional_feat_suffix', 'filter_inter_by_user_or_item', 'rm_dup_in

01 Oct 01:30    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 212
state = INFO
reproducibility = True
data_path = /Users/jakubmalczak/UNI/INŻ/SequentialRecommendation/recbole/data/dataset/Amazon_Sports_and_Outdoors
show_progress = True
save_dataset = False
save_dataloaders = False
benchmark_filename = None

Training Hyper Parameters:
checkpoint_dir = saved
epochs = 40
train_batch_size = 1024
learner = adam
learning_rate = 0.0001
eval_step = 2
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}
metrics = ['Recall', 'NDCG']
topk = [3, 5, 10, 20]
valid_metric = Recall@20
valid_metric_bigger = True
eval_batch_size = 256
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = rating
TIME_FIELD = timestamp
seq_len = Non

In [6]:
from recbole.data.utils import create_dataset

dataset = create_dataset(config)
logger.info(dataset)

01 Oct 01:33    INFO  Amazon_Sports_and_Outdoors
The number of users: 425813
Average actions of users: 8.514189360562877
The number of items: 1587220
Average actions of items: 22.46776812384576
The number of inters: 3625444
The sparsity of the dataset: 99.99946357975797%
Remain Fields: ['user_id', 'item_id', 'rating', 'timestamp', 'ent_id', 'ent_emb']


In [7]:
from recbole.data.utils import data_preparation

train_data, valid_data, test_data = data_preparation(config, dataset)

01 Oct 01:33    INFO  [Training]: train_batch_size = [1024] negative sampling: [None]
01 Oct 01:33    INFO  [Evaluation]: eval_batch_size = [256] eval_args: [{'split': {'LS': 'valid_and_test'}, 'group_by': 'user', 'order': 'TO', 'mode': 'full'}]


In [8]:
from recbole.utils.utils import get_model

model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
logger.info(model)

01 Oct 01:33    INFO  SASRecD(
  (item_embedding): Embedding(1587220, 256, padding_idx=0)
  (position_embedding): Embedding(50, 256)
  (feature_embed_layer_list): ModuleList(
    (0): Embedding(1587220, 128)
  )
  (trm_encoder): DIFTransformerEncoder(
    (layer): ModuleList(
      (0): DIFTransformerLayer(
        (multi_head_attention): DIFMultiHeadAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (query_p): Linear(in_features=256, out_features=256, bias=True)
          (key_p): Linear(in_features=256, out_features=256, bias=True)
          (query_layers): ModuleList(
            (0): Linear(in_features=128, out_features=128, bias=True)
          )
          (key_layers): ModuleList(
            (0): Linear(in_features=128, out_features=128, bias=True)
          )
          (fusion_layer): VanillaAtten

In [9]:
model.load_state_dict(data['state_dict'])
model.eval()

SASRecD(
  (item_embedding): Embedding(1587220, 256, padding_idx=0)
  (position_embedding): Embedding(50, 256)
  (feature_embed_layer_list): ModuleList(
    (0): Embedding(1587220, 128)
  )
  (trm_encoder): DIFTransformerEncoder(
    (layer): ModuleList(
      (0): DIFTransformerLayer(
        (multi_head_attention): DIFMultiHeadAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (query_p): Linear(in_features=256, out_features=256, bias=True)
          (key_p): Linear(in_features=256, out_features=256, bias=True)
          (query_layers): ModuleList(
            (0): Linear(in_features=128, out_features=128, bias=True)
          )
          (key_layers): ModuleList(
            (0): Linear(in_features=128, out_features=128, bias=True)
          )
          (fusion_layer): VanillaAttention(
            (pro

In [10]:
def predict_side_task(interaction, task):
        item_seq = interaction[model.ITEM_SEQ]
        item_seq_len = interaction[model.ITEM_SEQ_LEN]
        pos_items = interaction[model.POS_ITEM_ID]

        seq_output = model.forward(item_seq, item_seq_len)

        test_items_emb = model.item_embedding.weight
        scores = torch.matmul(seq_output, test_items_emb.transpose(0, 1))

        idx = torch.argmax(scores, dim=1)

        loss = dict()

        if model.attribute_predictor == '' or model.attribute_predictor == 'not':
            return {}

        if model.attribute_predictor == 'cos_sim':
            for i, a_predictor in enumerate(model.ap):
                true_emb = model.feature_embed_layer_list[i](idx if task == 3 else pos_items)
                pred_emb = a_predictor(seq_output)

                # Normalize embeddings to unit vectors
                pred_emb_norm = torch.nn.functional.normalize(pred_emb, p=2, dim=-1)
                true_emb_norm = torch.nn.functional.normalize(true_emb, p=2, dim=-1)

                # Compute cosine similarity
                cos_sim = (pred_emb_norm * true_emb_norm).sum(dim=-1)
                attribute_loss = (1 - cos_sim).mean()

                loss[model.selected_features[i]] = attribute_loss
        return loss

In [11]:
from tqdm import tqdm

task_3, task_4 = 0.0, 0.0

for i in tqdm(range(10)):
    interaction, _, _, _ = test_data._next_batch_data()
    task_3 += predict_side_task(interaction, task=3)['ent_emb']
    task_4 += predict_side_task(interaction, task=4)['ent_emb']

task_3 = task_3 / 10
task_4 = task_4 / 10
print(f"Wynik dla trzeciej kropki: {task_3}")
print(f"Wynik dla czwartej kropki: {task_4}")

100%|██████████| 10/10 [00:28<00:00,  2.83s/it]


Wynik dla trzeciej kropki: 0.7575549483299255
Wynik dla czwartej kropki: 0.8836170434951782
